# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "02_asset_characterization_Volatility"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-11"

# -------------------------
# Dynamic UPDATED
# -------------------------
from datetime import date
UPDATED = str(date.today())

PROJECT = "macro-market-analysis"
STAGE = "Research"
VERSION = "0.1.0"

# -------------------------
# Dynamic GIT HASH
# -------------------------
import subprocess

def get_git_hash():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"]
        ).decode("utf-8").strip()
    except:
        return ""

GIT_HASH = get_git_hash()

EXPERIMENT_ID = f"{NOTEBOOK_NAME}_{CREATED}_{VERSION}"

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
PURPOSE = """
Compute volatility features for each asset in the universe.

This notebook calculates time-series volatility metrics using
rolling dispersion (e.g. standard deviation) of returns
across multiple horizons.

The resulting features are stored **per asset** in:

data/features/asset/volatility/{asset}.parquet
"""

# ------------------------------------------------------------
# INPUT DATASETS
# ------------------------------------------------------------
# Updated: now each asset has its own processed parquet
INPUT_DATASETS = "data/processed/{asset}.parquet"

# ------------------------------------------------------------
# OUTPUT DATASETS
# ------------------------------------------------------------
OUTPUT_DATASETS = "data/features/asset/volatility/{asset}.parquet"

# ------------------------------------------------------------
# FEATURE FAMILY
# ------------------------------------------------------------
FEATURE_FAMILY = "volatility"

# ------------------------------------------------------------
# FEATURE DESCRIPTION
# ------------------------------------------------------------
FEATURE_DESCRIPTION = """
Volatility features quantify the dispersion of asset returns
across multiple time horizons.

They capture the intensity of market fluctuations and are
critical for:

- risk estimation
- regime detection (low vs high volatility environments)
- portfolio construction and allocation
- identifying stress or instability in markets

These features are purely time-series based and computed
independently per asset.
"""

# ============================================================
# Load ASSET_UNIVERSE from registry
# ============================================================

from quant_research.data.registry.universe_registry import ASSET_UNIVERSE

# Extraer solo los símbolos para iterar
ASSETS_UNIVERSE = [asset.symbol for asset in ASSET_UNIVERSE]

print("Assets in universe:", ASSETS_UNIVERSE)

# ------------------------------------------------------------
# DEPENDENCIES
# ------------------------------------------------------------
DEPENDENCIES = ["pandas >= 2.0", "numpy", "plotly >= 5.0", "pathlib"]

# ------------------------------------------------------------
# NOTES
# ------------------------------------------------------------
NOTES = """
Features are computed independently per asset.

Cross-sectional transformations such as:
    - rankings
    - z-scores
    - dispersion
are handled later in:

03_systemic_features.ipynb

Incremental feature saving per asset is planned to allow
recomputing only missing or updated rows.
"""

print(GIT_HASH)
print(UPDATED)

# 2. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

#-----------------------------
# Data manipulation
# -----------------------------
import pandas as pd  # pandas >= 2.0
import numpy as np   # numpy >= 1.25

# Optional: display settings for research notebooks
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.options.mode.chained_assignment = None  # avoid SettingWithCopyWarning

# ============================================================
# IMPORT PATHS
# ============================================================
from quant_research.config.paths import PROCESSED_DATA_PATH, MOMENTUM_FEATURE_PATH

# ============================================================
# Example: load processed parquet for one asset
# ============================================================
asset = "SPY"
processed_file = PROCESSED_DATA_PATH / f"{asset}.parquet"

df = pd.read_parquet(processed_file)
#print(df.head())

# ============================================================
# Example: output feature path for saving
# ============================================================
output_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"
print(output_file)

# -----------------------------
# Parquet engine
# -----------------------------
import pyarrow  # pyarrow >= 12.0

# -----------------------------
# Visualization
# -----------------------------
import plotly.express as px  # plotly >= 5.0
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# Optional future imports from src/ once implemented
# -----------------------------
# from src.utils import panel_utils, plotting_utils

# 3. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# -----------------------------
# Imports
# -----------------------------
from quant_research.config.paths import PROCESSED_DATA_PATH, VOLATILITY_FEATURE_PATH
from quant_research.data.registry.universe_registry import ASSET_UNIVERSE
# ------------------------------------------------------------
# FEATURE FAMILY
# ------------------------------------------------------------
FEATURE_FAMILY = "volatility"

# ------------------------------------------------------------
# FEATURE PARAMETERS
# ------------------------------------------------------------
LOOKBACK_WINDOWS = [5, 21, 63, 126, 252]
BASE_VOL_WINDOWS = [21, 63, 126, 252]

SMOOTH_WINDOWS = {
    5: None,
    21: 3,
    63: 3,
    126: 5,
    252: 5
}

DERIVATIVE_LAG = 5

VOL_SHORT_WINDOW = 21
VOL_MEDIUM_WINDOW = 63
VOL_LONG_WINDOW = 126

# ------------------------------------------------------------
# VOLATILITY OF VOLATILITY
# ------------------------------------------------------------
VOV_WINDOWS = [21, 63]

# ------------------------------------------------------------
# VOL TERM STRUCTURE
# ------------------------------------------------------------
VOL_TERM_STRUCTURE_CONFIG = {
    "VOL_TS_21_63": (21, 63),
    "VOL_TS_63_126": (63, 126),
}

# ------------------------------------------------------------
# VOL EXPANSION
# ------------------------------------------------------------
EXPANSION_PAIRS = [
    (21, 63),
    (21, 126),
    (63, 126),
]

# ------------------------------------------------------------
# VOL REGIME PARAMETERS
# ------------------------------------------------------------
Z_SCORE_WINDOW = 252
PERCENTILE_WINDOW = 252

# ------------------------------------------------------------
# AGGREGATION
# ------------------------------------------------------------
VSI_WINDOWS = [21, 63, 126]

# ------------------------------------------------------------
# OPTIONAL FEATURES
# ------------------------------------------------------------
INCLUDE_DOWNSIDE_VOL = False
# ------------------------------------------------------------
# UNIVERSE CONFIGURATION
# ------------------------------------------------------------
# Use ASSET_UNIVERSE from registry
ASSETS = [asset.symbol for asset in ASSET_UNIVERSE]

# ------------------------------------------------------------
# OUTPUT CONFIGURATION
# ------------------------------------------------------------
EXPORT_FORMAT = "parquet"
OVERWRITE_EXISTING = True

# ------------------------------------------------------------
# RESEARCH OPTIONS
# ------------------------------------------------------------
# Sample assets for visualizations and sanity checks
SAMPLE_ASSETS_FOR_PLOTS = [
    "BTC",
    "SPY"
]

# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------
RANDOM_SEED = 42

# 4. Data Loading

In [ ]:
# ============================================================
# 4. DATA LOADING
# ============================================================

import pandas as pd

# Diccionario para guardar los DataFrames de INPUT (read-only)
asset_data = {}

print("Loading processed data for each asset...\n")

for asset in ASSETS:
    processed_file = PROCESSED_DATA_PATH / f"{asset}.parquet"
    
    if processed_file.exists():
        df = pd.read_parquet(processed_file)
        
        # -----------------------------
        # Validations
        # -----------------------------
        
        # Ensure datetime index
        if not pd.api.types.is_datetime64_any_dtype(df.index):
            try:
                df.index = pd.to_datetime(df.index)
            except Exception as e:
                raise ValueError(f"Asset {asset}: Cannot convert index to datetime") from e
        
        # Sort index ascending
        df = df.sort_index()
        
        # Drop duplicated indices
        df = df[~df.index.duplicated(keep="first")]
        
        # -----------------------------
        # Required columns (STRICT)
        # -----------------------------
        required_cols = ["adj_close", "log_ret"]
        
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Asset {asset}: Missing required columns: {missing_cols}")
        
        # -----------------------------
        # Optional columns (for future features)
        # -----------------------------
        optional_cols = ["High", "Low"]
        has_ohlc = all(col in df.columns for col in optional_cols)
        
        # Flags (metadata, no contaminación de columnas)
        df.attrs["feature_flags"] = {
            "has_ohlc": has_ohlc
        }
        
        # -----------------------------
        # Save as INPUT (READ-ONLY)
        # -----------------------------
        asset_data[asset] = df
        
        print(
            f"Loaded {asset} → {len(df)} rows, {df.shape[1]} columns | "
            f"OHLC: {'YES' if has_ohlc else 'NO'}"
        )
    
    else:
        print(f"WARNING: Processed parquet not found for {asset}: {processed_file}")
        
print("\nAll assets loaded. Ready for volatility feature computation.")

# 5. Core Computations  

Volatility Feature Review

Volatility measures the magnitude of price fluctuations of an asset over time.  
It captures the risk or uncertainty associated with the asset's returns and is a fundamental component in quantitative finance.

**Why we compute volatility features:**

- **Risk Assessment:** High volatility signals greater uncertainty and potential drawdowns, while low volatility implies more stable price behavior.  
- **Strategy Design:** Volatility is used for position sizing, risk-adjusted returns, and adaptive strategies.  
- **Cross-Asset Analysis:** Comparing volatility across assets helps detect market regimes and relative risk levels.  
- **Feature Engineering:** Volatility features serve as inputs for allocation models, regime detection, and systemic market analysis.


## 5.1 Volatility Level

Annualized volatility provides a long-term estimate of risk,
but financial markets do not exhibit constant volatility.

Instead, volatility tends to cluster through time,
a phenomenon known as volatility clustering.

Volatility level measures how risk evolves dynamically
by computing volatility over a moving time window.

Mathematically:

$$
\sigma_t = Std(r_{t-n}, ..., r_t)
$$

Where:

- $n$ is the rolling window length
- $r_t$ represents asset returns

The rolling volatility is then annualized:

$$
\sigma_{annual,t} = \sigma_t \times \sqrt{252}
$$

This metric allows us to identify:

- Risk expansion periods
- Market stress events
- Volatility regimes
- Cross-asset risk transmission

Rising volatility often precedes or accompanies market drawdowns,
making it a key indicator for regime detection.

In [ ]:
# ============================================================
# 5. FEATURE DATA CONTAINERS
# ============================================================

feature_dfs = {}

for asset, data_df in asset_data.items():
    
    # Compartimos index (NO copy → más eficiente)
    feature_df = pd.DataFrame(index=data_df.index)
    
    feature_dfs[asset] = feature_df

print("Feature containers initialized.\n")

In [ ]:
# ============================================================
# 5.1 VOLATILITY (VOL_h)
# ============================================================

print("Computing Volatility features...\n")

for asset in ASSETS:
    
    data_df = asset_data[asset]
    feat_df = feature_dfs[asset]
    
    ret = data_df["log_ret"]
    
    for h in LOOKBACK_WINDOWS:
        
        vol_col = f"VOL_{h}"
        
        # Rolling volatility (std of log returns)
        feat_df[vol_col] = ret.rolling(window=h).std()
    
    print(f"{asset}: VOL features computed")

print("\nVolatility computation completed.")

In [ ]:
feature_df.columns

## 5.2 Volatility Dynamics (VEL, VEL_S, ACC, ACC_S)

While the volatility level (`VOL_N`) measures the magnitude of price fluctuations over a given horizon, it does not capture how volatility itself evolves through time.

To better understand volatility behavior, we compute **volatility dynamics**, which describe the **rate and acceleration of change in volatility**.

### Velocity (VEL)

**Volatility Velocity (`VOL_N_VEL`)** measures the **first derivative of volatility**, i.e., the day-to-day change in volatility:

\[
VEL_t = VOL_t - VOL_{t-1}
\]

This feature captures whether volatility is **increasing or decreasing**.  
Rising velocity may signal **volatility expansion**, while negative values indicate **volatility compression**.

### Smoothed Velocity (VEL_S)

Financial time series are often noisy. To reduce short-term fluctuations, we compute a **smoothed velocity (`VOL_N_VEL_S`)** using a short rolling average.

This helps identify **persistent volatility trends** rather than transient spikes.

### Acceleration (ACC)

**Volatility Acceleration (`VOL_N_ACC`)** represents the **second derivative of volatility**, computed as the change in velocity:

\[
ACC_t = VEL_t - VEL_{t-1}
\]

Acceleration captures **how quickly volatility dynamics themselves are changing**, which can signal **early shifts in market regimes**.

### Smoothed Acceleration (ACC_S)

A smoothed version (`VOL_N_ACC_S`) is also computed to reduce noise and highlight more stable structural changes in volatility dynamics.

---

### Interpretation

These features provide complementary information about volatility:

| Feature | Interpretation |
|-------|------|
| `VOL_N` | Absolute volatility level |
| `VOL_N_VEL` | Change in volatility (expansion vs compression) |
| `VOL_N_VEL_S` | Smoothed volatility trend |
| `VOL_N_ACC` | Change in volatility velocity |
| `VOL_N_ACC_S` | Smoothed structural shifts in volatility dynamics |

Together, these features allow the model to capture **not only how volatile an asset is, but also how volatility is evolving through time**.

In [ ]:
# ============================================================
# 5.3 VOLATILITY DYNAMICS (VEL / ACC)
# ============================================================

print("Computing Volatility dynamics (VEL / ACC)...\n")

DERIVATIVE_LAG = 1  # mantenemos parametrizable

for asset in ASSETS:
    
    feat_df = feature_dfs[asset]
    
    for h in LOOKBACK_WINDOWS:
        
        vol_col = f"VOL_{h}"
        
        vel_col = f"{vol_col}_VEL"
        acc_col = f"{vol_col}_ACC"
        
        vel_s_col = f"{vel_col}_S"
        acc_s_col = f"{acc_col}_S"
        
        if vol_col not in feat_df.columns:
            print(f"{asset}: Missing {vol_col} for dynamics")
            continue
        
        vol_series = feat_df[vol_col]
        
        # -----------------------------
        # VELOCITY
        # -----------------------------
        feat_df[vel_col] = vol_series.diff(DERIVATIVE_LAG)
        
        # -----------------------------
        # ACCELERATION
        # -----------------------------
        feat_df[acc_col] = feat_df[vel_col].diff(DERIVATIVE_LAG)
        
        # -----------------------------
        # SMOOTHING
        # -----------------------------
        smooth_window = SMOOTH_WINDOWS.get(h, None)
        
        if smooth_window is not None:
            
            feat_df[vel_s_col] = (
                feat_df[vel_col]
                .rolling(smooth_window)
                .mean()
            )
            
            feat_df[acc_s_col] = (
                feat_df[acc_col]
                .rolling(smooth_window)
                .mean()
            )
    
    print(f"{asset}: Volatility dynamics computed")

print("\nVolatility dynamics completed.")

In [ ]:
feature_df.columns

## 5.3 Volatility Normalization

Raw volatility levels are difficult to interpret across time because financial markets naturally move through different volatility environments.

For example, a volatility level of 20% may represent **extreme stress in one regime** but **normal conditions in another**, depending on the historical context.

To make volatility more comparable through time, we construct **Volatility Normalization** based on normalization and relative positioning.

These indicators aim to answer questions such as:

- Is volatility currently high relative to its historical distribution?
- Is volatility expanding or compressing relative to its past behavior?
- Is the market entering a high-risk regime?

To capture these dynamics, we construct three complementary indicators:

1. **Volatility Z-Score** — measures how many standard deviations current volatility is from its historical mean.

2. **Volatility Percentile** — measures the relative rank of current volatility within its historical distribution.

3. **Volatility Expansion / Compression** — identifies whether volatility is increasing or decreasing relative to its recent history.

These indicators transform raw volatility into **regime-aware signals**, which are often more informative for models focused on **risk management, asset allocation, and regime detection**.

In [ ]:
# ============================================================
# 5.2 VOLATILITY NORMALIZATION (Z-SCORE + PERCENTILE)
# ============================================================

print("Computing Volatility normalization features...\n")

for asset in ASSETS:
    
    feat_df = feature_dfs[asset]
    
    for h in LOOKBACK_WINDOWS:
        
        vol_col = f"VOL_{h}"
        z_col   = f"{vol_col}_Z"
        pctl_col = f"{vol_col}_PCTL"
        
        if vol_col not in feat_df.columns:
            print(f"{asset}: Missing {vol_col} for normalization")
            continue
        
        vol_series = feat_df[vol_col]
        
        # -----------------------------
        # Z-SCORE
        # -----------------------------
        rolling_mean = vol_series.rolling(Z_SCORE_WINDOW).mean()
        rolling_std  = vol_series.rolling(Z_SCORE_WINDOW).std()
        
        feat_df[z_col] = (vol_series - rolling_mean) / rolling_std
        
        # -----------------------------
        # PERCENTILE (rolling rank)
        # -----------------------------
        feat_df[pctl_col] = (
            vol_series
            .rolling(PERCENTILE_WINDOW)
            .rank(pct=True)
        )
    
    print(f"{asset}: VOL normalization computed")

print("\nVolatility normalization completed.")

## 5.4 Volatility of Volatility

While volatility measures the magnitude of market fluctuations, it is also useful to understand **how stable volatility itself is over time**.

This concept is known as **Volatility of Volatility (VoV)**.

VoV measures the dispersion of the volatility process, capturing periods where volatility becomes **unstable or erratic**, often associated with **market regime transitions**.

---

### Definition

Volatility of Volatility is defined as the rolling standard deviation of a volatility series using the **same lookback horizon**:

$$
VOV_{a,t} = \sigma \left( VOL_{a,t-i} \right), \quad i = 1, \dots, a
$$

where:

- $VOL_a$ is the volatility computed using lookback window $a$
- $\sigma(\cdot)$ denotes the standard deviation

In expanded form:

$$
VOV_{a,t} =
\sqrt{
\frac{1}{a}
\sum_{i=1}^{a}
\left(
VOL_{a,t-i} - \overline{VOL}_{a,t}
\right)^2
}
$$

---

### Normalization

To make Volatility of Volatility comparable across time, a normalized version is computed using a rolling Z-score:

$$
VOV_{a,t}^{Z} =
\frac{
VOV_{a,t} - \mu_{t}^{(Z)}
}{
\sigma_{t}^{(Z)}
}
$$

where:

- $\mu_{t}^{(Z)}$ is the rolling mean of $VOV_{a,t}$ over a long-term window (e.g., 252 days)
- $\sigma_{t}^{(Z)}$ is the rolling standard deviation over the same window

---

### Interpretation

Volatility of Volatility captures the **stability of the volatility regime**.

| Feature | Interpretation |
|--------|---------------|
| `VOV_21` | stability of short-term volatility |
| `VOV_63` | stability of medium-term volatility |
| `VOV_21_Z` | relative instability vs history |
| `VOV_63_Z` | relative instability vs history |

Typical interpretations include:

- **High VoV**  
  Volatility is unstable and rapidly changing, often signaling **regime transitions** or structural uncertainty.

- **Low VoV**  
  Volatility is stable, suggesting a **persistent regime**, either calm or sustained high-volatility conditions.

- **High VoV Z-score**  
  Current instability is extreme relative to history → strong signal of **regime shift**.

- **Low VoV Z-score**  
  Volatility regime is unusually stable.

---

### Design Note

This implementation uses a **single-horizon definition**:

$$
VOV_a = std(VOL_a, a)
$$

This ensures:

- conceptual consistency with the underlying volatility measure  
- no additional hyperparameters  
- direct interpretability across horizons  

The Z-score normalization aligns VoV with the rest of the feature framework, enabling **cross-time comparability and regime detection**.

In [ ]:
# ============================================================
# 5.4 VOLATILITY OF VOLATILITY (VOV)
# ============================================================

print("Computing Volatility of Volatility (VOV)...\n")

for asset in ASSETS:
    
    feat_df = feature_dfs[asset]
    
    for h in LOOKBACK_WINDOWS:
        
        vol_col = f"VOL_{h}"
        vov_col = f"VOV_{h}"
        vov_z_col = f"{vov_col}_Z"
        
        if vol_col not in feat_df.columns:
            print(f"{asset}: Missing {vol_col} for VOV")
            continue
        
        vol_series = feat_df[vol_col]
        
        # -----------------------------
        # VOV (std of volatility)
        # -----------------------------
        feat_df[vov_col] = vol_series.rolling(window=h).std()
        
        # -----------------------------
        # VOV Z-score
        # -----------------------------
        rolling_mean = feat_df[vov_col].rolling(Z_SCORE_WINDOW).mean()
        rolling_std  = feat_df[vov_col].rolling(Z_SCORE_WINDOW).std()
        
        feat_df[vov_z_col] = (feat_df[vov_col] - rolling_mean) / rolling_std
    
    print(f"{asset}: VOV computed")

print("\nVolatility of Volatility completed.")

## 5.5 Volatility Term Structure & Expansion

Volatility is not only defined by its level, but also by its **structure across time horizons**.

The **Volatility Term Structure (VOL_TS)** captures the relationship between short-term and longer-term volatility, providing insight into **risk expectations and regime dynamics**.

---

### Definition

The term structure is defined as the difference between two volatility horizons:

$$
VOL\_TS_{a,b,t} = VOL_{a,t} - VOL_{b,t}
$$

where:

- $a < b$
- $VOL_a$ = short-term volatility
- $VOL_b$ = longer-term volatility

---

### Normalization

To make the term structure comparable over time, a rolling Z-score is applied:

$$
VOL\_TS_{a,b,t}^{Z} =
\frac{
VOL\_TS_{a,b,t} - \mu_t
}{
\sigma_t
}
$$

where:

- $\mu_t$ = rolling mean of $VOL\_TS_{a,b,t}$ over a long-term window (e.g., 252)
- $\sigma_t$ = rolling standard deviation over the same window

---

### Volatility Expansion

Volatility expansion captures whether short-term volatility exceeds longer-term volatility, indicating **increasing short-term risk pressure**.

---

### Definition

$$
EXPANSION_{a,b,t} =
\begin{cases}
1 & \text{if } VOL_{a,t} > VOL_{b,t} \\
0 & \text{otherwise}
\end{cases}
$$

---

### Interpretation

Volatility term structure provides information about **how risk is distributed across time horizons**:

- **$VOL\_TS_{a,b} > 0$ (front-loaded volatility)**  
  Short-term volatility is higher than long-term volatility → **market stress or shock**

- **$VOL\_TS_{a,b} < 0$ (back-loaded volatility)**  
  Long-term volatility dominates → **persistent uncertainty**

Volatility expansion acts as a **binary regime indicator**:

- $EXPANSION = 1$ → volatility is increasing in the short term (**risk acceleration**)  
- $EXPANSION = 0$ → no short-term dominance  

---

### Design Notes

- Uses **difference (not ratio)** for robustness across assets  
- Expansion is a **state variable**, not a magnitude measure  
- Complements:
  - VOL (level)
  - VEL/ACC (dynamics)
  - VOV (stability)

In [ ]:
# ============================================================
# 5.5 VOLATILITY TERM STRUCTURE & EXPANSION
# ============================================================

print("Computing Volatility Term Structure and Expansion...\n")

for asset in ASSETS:
    
    feat_df = feature_dfs[asset]
    
    for (short_w, long_w) in EXPANSION_PAIRS:
        
        short_col = f"VOL_{short_w}"
        long_col  = f"VOL_{long_w}"
        
        ts_col = f"VOL_TS_{short_w}_{long_w}"
        ts_z_col = f"{ts_col}_Z"
        exp_col = f"EXP_{short_w}_{long_w}"
        
        if short_col not in feat_df.columns or long_col not in feat_df.columns:
            print(f"{asset}: Missing VOL columns for TS {short_w}-{long_w}")
            continue
        
        # -----------------------------
        # TERM STRUCTURE (difference)
        # -----------------------------
        feat_df[ts_col] = feat_df[short_col] - feat_df[long_col]
        
        # -----------------------------
        # Z-SCORE
        # -----------------------------
        rolling_mean = feat_df[ts_col].rolling(Z_SCORE_WINDOW).mean()
        rolling_std  = feat_df[ts_col].rolling(Z_SCORE_WINDOW).std()
        
        feat_df[ts_z_col] = (feat_df[ts_col] - rolling_mean) / rolling_std
        
        # -----------------------------
        # EXPANSION FLAG
        # -----------------------------
        feat_df[exp_col] = (
            feat_df[short_col] > feat_df[long_col]
        ).astype(int)
    
    print(f"{asset}: Term Structure + Expansion computed")

print("\nVolatility Term Structure & Expansion completed.")

In [ ]:
feature_df.columns

#### Volatility Z-Score

The Volatility Z-Score measures how extreme the current volatility level is relative to its historical distribution.

Formally:

$$
ZVOL_{w,t} =
\frac{
VOL_{w,t} - \mu_{w,t}
}{
\sigma_{w,t}
}
$$

where:

- $VOL_{w,t}$ is the volatility computed using lookback window $w$
- $\mu_{w,t}$ is the rolling mean of the volatility series
- $\sigma_{w,t}$ is the rolling standard deviation of the volatility series

### Interpretation

| Z-Score | Interpretation |
|------|------|
| > 2 | extremely high volatility |
| 1 to 2 | elevated volatility |
| -1 to 1 | normal regime |
| < -1 | unusually low volatility |

Z-scores are widely used in quantitative models because they **normalize features across time**, making them more comparable across different volatility regimes.

#### Volatility Percentile

While Z-scores measure how many standard deviations the current volatility is from its mean, another useful normalization approach is to compute the **percentile rank of volatility within its historical distribution**.

The Volatility Percentile measures the **relative position of the current volatility level within a rolling historical window**.

Formally, the percentile rank can be expressed as:

$$
VPERC_{w,t} =
\frac{
\#\{VOL_{w,i} \leq VOL_{w,t}\}
}{
N
}
$$

where:

- $VOL_{w,t}$ is the volatility computed with lookback window $w$
- $N$ is the number of observations in the rolling window
- $\#\{\cdot\}$ counts the number of past observations less than or equal to the current value

The resulting value lies between:

$$
0 \le VPERC_{w,t} \le 1
$$

### Interpretation

| Percentile | Interpretation |
|------|------|
| > 0.9 | extremely high volatility regime |
| 0.7 – 0.9 | elevated volatility |
| 0.3 – 0.7 | normal regime |
| < 0.3 | low volatility regime |

Percentiles are particularly useful because they **do not assume normality of the volatility distribution**, making them robust for financial time series where volatility often exhibits skewness and heavy tails.

In [ ]:
feature_df.columns

## 5.6 Volatility Strength Index (VSI)

While individual volatility measures capture risk at specific horizons, it is useful to construct a **composite indicator** summarizing the overall volatility environment.

This is the **Volatility Strength Index (VSI)**.

---

### Definition

VSI is defined as the average of normalized volatility measures across multiple horizons:

$$
VSI_t = \frac{1}{N} \sum_{a \in H} VOL_{a,t}^{Z}
$$

where:

- $H$ = set of horizons (e.g. 21, 63, 126)
- $VOL_a^Z$ = Z-score normalized volatility
- $N$ = number of horizons

---

### Optional Smoothing

To reduce noise:

$$
VSI_t^{S} = SMA(VSI_t, k)
$$

where:

- $k$ = smoothing window (e.g. 3 or 5)

---

### Interpretation

| Feature | Interpretation |
|--------|---------------|
| `VSI` | aggregated volatility regime |
| `VSI_S` | smoothed regime signal |

- **High VSI**  
  Broad-based elevated volatility across horizons  
  → **risk-off regime**

- **Low VSI**  
  Compressed volatility  
  → **risk-on / calm regime**

---

### Regime Classification (Optional)

You may define regimes using thresholds:

- $VSI > +1$ → High volatility regime  
- $VSI < -1$ → Low volatility regime  

---

### Design Notes

- Direct analogue to **Momentum Strength Index (MSI)**
- Uses **Z-normalized inputs** → scale invariant
- Avoids dominance of any single horizon
- Captures **systemic volatility conditions**

---

### Implementation Choices

Typical horizon set:

$$
H = \{21, 63, 126\}
$$

Alternative:

- include 252 for long-term regime
- exclude 5 to avoid noise dominance

### Extension: Weighted VSI (Future Work)

The VSI can be extended to a weighted formulation:

$$
VSI_t = \sum_{a \in H} w_a \cdot VOL_{a,t}^{Z}
$$

where:

- $w_a$ are weights assigned to each horizon
- $\sum w_a = 1$

This allows emphasizing specific horizons (e.g., short-term risk vs long-term risk).

In the current implementation, equal weights are used:

$$
w_a = \frac{1}{N}
$$

In [ ]:
# ============================================================
# 5.6 VOLATILITY STRENGTH INDEX (VSI)
# ============================================================

print("Computing Volatility Strength Index (VSI)...\n")

# Horizontes incluidos en el índice
VSI_HORIZONS = [21, 63, 126]

for asset in ASSETS:
    
    feat_df = feature_dfs[asset]
    
    vsi_components = []
    
    # -----------------------------
    # COLLECT VOL_Z COMPONENTS
    # -----------------------------
    for h in VSI_HORIZONS:
        
        col = f"VOL_{h}_Z"
        
        if col not in feat_df.columns:
            print(f"{asset}: Missing {col} for VSI")
            continue
        
        vsi_components.append(feat_df[col])
    
    if len(vsi_components) == 0:
        print(f"{asset}: No valid components for VSI")
        continue
    
    # -----------------------------
    # VSI (simple average)
    # -----------------------------
    feat_df["VSI"] = sum(vsi_components) / len(vsi_components)
    
    # -----------------------------
    # OPTIONAL SMOOTHING
    # -----------------------------
    smooth_window = SMOOTH_WINDOWS.get(63, None)
    
    if smooth_window is not None:
        feat_df["VSI_S"] = (
            feat_df["VSI"]
            .rolling(smooth_window)
            .mean()
        )
    
    print(f"{asset}: VSI computed")

print("\nVolatility Strength Index completed.")

# 6. Feature Export

In [ ]:
feature_df.columns

In [ ]:
data_df.columns

In [ ]:
# ============================================================
# 6. FEATURE EXPORT
# ============================================================

print("Exporting volatility features...\n")

for asset in ASSETS:
    
    feat_df = feature_dfs[asset]
    
    # -----------------------------
    # Safety checks
    # -----------------------------
    
    if feat_df.empty:
        print(f"{asset}: WARNING → empty feature_df, skipping")
        continue
    
    if not pd.api.types.is_datetime64_any_dtype(feat_df.index):
        raise ValueError(f"{asset}: feature_df index is not datetime")
    
    # Ordenar (defensivo)
    feat_df = feat_df.sort_index()
    
    # -----------------------------
    # File path (from config)
    # -----------------------------
    
    file_path = VOLATILITY_FEATURE_PATH / f"{asset}.parquet"
    
    # -----------------------------
    # Export
    # -----------------------------
    
    feat_df.to_parquet(file_path)
    
    print(
        f"{asset}: exported → {file_path.name} | "
        f"{len(feat_df)} rows, {feat_df.shape[1]} features"
    )

print("\nVolatility feature export completed.")

# 7. Visualization

In [ ]:
import pandas as pd
import plotly.graph_objects as go

ASSET = "BTC"
data_df = pd.read_parquet(PROCESSED_DATA_PATH / f"{ASSET}.parquet")

file_path = VOLATILITY_FEATURE_PATH / f"{ASSET}.parquet"
df = pd.read_parquet(file_path)

df.tail()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["VOL_21_Z"],
    name="VOL_21_Z"
))

fig.add_trace(go.Scatter(
    x=df.index, y=df["VOV_21_Z"],
    name="VOV_21_Z"
))

fig.update_layout(
    title=f"{ASSET} — VOL vs VOV (Z-score)",
    template="plotly_dark",
    height=500
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["VOV_21"],
    name="VOV_21"
))

fig.add_trace(go.Scatter(
    x=df.index, y=df["VOV_63"],
    name="VOV_63"
))

fig.update_layout(
    title=f"{ASSET} — Volatility of Volatility",
    template="plotly_dark"
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["VOL_21_Z"],
    name="VOL_21_Z"
))

# líneas de referencia
fig.add_hline(y=0)
fig.add_hline(y=2, line_dash="dash")
fig.add_hline(y=-2, line_dash="dash")

fig.update_layout(
    title=f"{ASSET} — Volatility Z-score",
    template="plotly_dark"
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["VOL_TS_21_63"],
    name="TS 21-63"
))

fig.add_trace(go.Scatter(
    x=df.index, y=df["VOL_TS_21_63_Z"],
    name="TS Z"
))

fig.add_hline(y=0)

fig.update_layout(
    title=f"{ASSET} — Volatility Term Structure",
    template="plotly_dark"
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df["EXP_21_63"],
    fill="tozeroy",
    name="Expansion",
    mode="lines"
))

fig.update_layout(
    title=f"{ASSET} — Volatility Expansion Regime",
    template="plotly_dark"
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["VSI"],
    name="VSI"
))

fig.add_trace(go.Scatter(
    x=df.index, y=df["VSI_S"],
    name="VSI_S"
))

fig.update_layout(
    title=f"{ASSET} — Volatility Strength Index",
    template="plotly_dark"
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df["VSI"],
    name="VSI",
    yaxis="y1"
))

fig.add_trace(go.Scatter(
    x=data_df.index,
    y=data_df["log_ret"].rolling(21).sum(),
    name="21d return",
    yaxis="y2"
))

fig.update_layout(
    title=f"{ASSET} — VSI vs Returns",
    template="plotly_dark",
    
    yaxis=dict(title="VSI"),
    yaxis2=dict(
        title="Returns",
        overlaying="y",
        side="right"
    )
)

fig.show()

# 8. Feature_Engine Validation (For Update in Stage 2)

In [ ]:
ASSET = "BTC"

data_df = asset_data[ASSET]

In [ ]:
from quant_research.config.paths import VOLATILITY_FEATURE_PATH

asset_file = VOLATILITY_FEATURE_PATH / f"{ASSET}.parquet"

In [ ]:
from quant_research.features.asset.volatility.feature_validator import validate_asset_features

result = validate_asset_features(
    asset=ASSET,
    df_raw=data_df,      # 👈 INPUT para reconstruir
    asset_file=asset_file,
    verbose=True
)

result

In [ ]:
data_df.columns

In [ ]:
feature_df.columns

In [ ]:
from quant_research.features.asset.volatility.feature_validator import validate_universe_features

summary = validate_universe_features(
    asset_dfs=asset_data,
    feature_path=VOLATILITY_FEATURE_PATH,
    verbose=False
)

summary